# Practical: score-based generative modelling on a mixture of two Gaussians

In the lecture we saw that **Langevin dynamics gets the proportions wrong** when the data has several well-separated modes, and that **annealed Langevin dynamics** (many noise levels, from large to small) fixes it.

In this practical you reproduce that experiment on a 2-D toy problem, first with the *exact* score, then with a score **learned by denoising score matching**:

1. sample from a mixture of two Gaussians with unequal weights,
2. compute its score and look at it,
3. sample with Langevin dynamics, and observe the wrong proportions,
4. sample with annealed Langevin dynamics, and observe the right proportions,
5. learn the score with denoising score matching: first at a single noise level, then with a noise-conditional score network (NCSN).

Cells marked **Exercise** contain a `TODO` for you to fill in. Each one is followed by a **check** cell: run it, and it either prints `OK` or tells you what is wrong (usually a shape).

*Requirements:* `torch`, `numpy`, `matplotlib`, and the file **`utils_score_matching.py`**, which contains the helpers: the mixture, the measurements, the plots and the checks. You do not need to read it. Keep it in the same folder as this notebook; on Google Colab, the first cell downloads it. Everything runs on a CPU in under a minute.

In [ ]:
import math
import os
import urllib.request

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# The helpers are in utils_score_matching.py. If it is not next to this notebook (e.g. on Colab), download it.
UTILS_URL = "https://raw.githubusercontent.com/franciscoandrade90/deep-learning-practicals/main/score_matching/utils_score_matching.py"
if not os.path.exists("utils_score_matching.py"):
    urllib.request.urlretrieve(UTILS_URL, "utils_score_matching.py")
from utils_score_matching import *

torch.manual_seed(0)
np.random.seed(0)

## 1. The target distribution

We take
$$
p(\mathbf{x}) = \pi_1\,\mathcal{N}(\mathbf{x};\boldsymbol{\mu}_1,\mathbf{I}) + \pi_2\,\mathcal{N}(\mathbf{x};\boldsymbol{\mu}_2,\mathbf{I}),
\qquad
\pi_1 = 0.2,\ \pi_2 = 0.8,
\qquad
\boldsymbol{\mu}_1 = (-5,-5),\ \boldsymbol{\mu}_2 = (5,5).
$$
The small mode (bottom left) holds 20% of the mass, the large one (top right) 80%.

The weights $\pi_k$ and the means $\boldsymbol{\mu}_k$ are given in `utils_score_matching.py`, as `PIS` (shape `(2,)`) and `MUS` (shape `(2, 2)`, one mean per row):

In [ ]:
print("PIS =", PIS)
print("MUS =", MUS)

**Exercise 1.** Write `sample_data(n)`, which returns `n` samples from $p$ as a tensor of shape `(n, 2)`.

*Hint:* first draw which component each sample comes from (`torch.multinomial`), then add standard Gaussian noise to its mean.

In [ ]:
def sample_data(n):
    '''Returns n samples from p, shape (n, 2).'''
    # TODO
    raise NotImplementedError

In [ ]:
check_sample_data(sample_data)

We also need a way to measure the proportions of a set of samples. Two helpers are given in `utils_score_matching.py`. The two modes are separated by the line $x_1 + x_2 = 0$, and `fraction_small_mode(x)` returns the fraction of the samples `x` on the side of the small mode. `plot_samples(x, ax=None, title=None)` draws the samples over the density of $p$; by default, its title gives that fraction.

In [ ]:
x_data = sample_data(10_000)   # our "training set"
plot_samples(x_data[:2000], title=f"data: {fraction_small_mode(x_data):.0%} in the small mode")
plt.show()

## 2. The score of the mixture

For a mixture of Gaussians with identity covariance, the score is a weighted average of the scores of the components:
$$
\nabla_\mathbf{x} \log p(\mathbf{x}) = w_1(\mathbf{x})\,(\boldsymbol{\mu}_1 - \mathbf{x}) + w_2(\mathbf{x})\,(\boldsymbol{\mu}_2 - \mathbf{x}),
$$
with
$$
w_k(\mathbf{x}) = \frac{\pi_k\,\mathcal{N}(\mathbf{x};\boldsymbol{\mu}_k,\mathbf{I})}{\pi_1\,\mathcal{N}(\mathbf{x};\boldsymbol{\mu}_1,\mathbf{I}) + \pi_2\,\mathcal{N}(\mathbf{x};\boldsymbol{\mu}_2,\mathbf{I})},
\qquad k = 1, 2,
$$
where $w_k(\mathbf{x})$ is the probability that $\mathbf{x}$ came from component $k$ (so $w_1 + w_2 = 1$).

If we add Gaussian noise of standard deviation $\sigma$ to the data, the noisy distribution $p_\sigma$ is again a mixture of Gaussians, with the same weights and means, but with covariance $(1+\sigma^2)\,\mathbf{I}$. Its score is
$$
\nabla_\mathbf{x} \log p_\sigma(\mathbf{x}) = \frac{w_1^\sigma(\mathbf{x})\,(\boldsymbol{\mu}_1 - \mathbf{x}) + w_2^\sigma(\mathbf{x})\,(\boldsymbol{\mu}_2 - \mathbf{x})}{1+\sigma^2},
$$
with $w_k^\sigma$ defined as above but with covariance $(1+\sigma^2)\,\mathbf{I}$.

The weights $w_k^\sigma(\mathbf{x})$ are given (they are computed with a softmax, which is numerically stable):

In [ ]:
def component_weights(x, sigma=0.0):
    '''x: (n, 2); sigma: a number, or a tensor of shape (n, 1).
    Returns w of shape (n, 2), where w[:, k] is the weight w_k(x) of component k.'''
    var = 1.0 + sigma**2
    logits = [torch.log(PIS[k]) - 0.5 * ((x - MUS[k]) ** 2).sum(dim=1, keepdim=True) / var
              for k in range(2)]                                   # two tensors of shape (n, 1)
    return torch.softmax(torch.cat(logits, dim=1), dim=1)          # (n, 2)

**Exercise 2.** Complete `true_score(x, sigma=0.0)`, which returns $\nabla_\mathbf{x} \log p_\sigma(\mathbf{x})$, using the formula above.

*Shapes:* `x` is `(n, 2)` (one point per row), `MUS[0]` and `MUS[1]` are `(2,)`, so `MUS[0] - x` is `(n, 2)`. The weights `w1` and `w2` are `(n, 1)`: one weight per point, which multiplies the whole row of that point.

In [ ]:
def true_score(x, sigma=0.0):
    '''x: (n, 2); sigma: a number, or a tensor of shape (n, 1). Returns the score, shape (n, 2).'''
    var = 1.0 + sigma**2
    w1, w2 = component_weights(x, sigma).split(1, dim=1)            # each (n, 1)
    # TODO
    raise NotImplementedError

In [ ]:
check_true_score(true_score)   # compares with automatic differentiation of the log-density

In [ ]:
# plot_field(score_fn, ax, title, sigma), from utils_score_matching.py, draws the directions of the arrows
# over the density of p_sigma
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, s in zip(axes, [0.0, 3.0, 10.0]):
    plot_field(lambda x: true_score(x, s), ax=ax, sigma=s, title=f"score of $p_\\sigma$, $\\sigma$ = {s}")
plt.show()

**Question.** Look at the score of $p$ ($\sigma = 0$), far from the line $x_1 + x_2 = 0$. Does it depend on the weights $\pi_1, \pi_2$? What does that suggest for a sampler that only uses the score?

## 3. Langevin dynamics with the exact score

Langevin dynamics, as in the lecture:
$$
\mathbf{x}_{k+1} = \mathbf{x}_k + \epsilon\,\nabla_\mathbf{x}\log p(\mathbf{x}_k) + \sqrt{2\epsilon}\,\mathbf{z}_k,
\qquad \mathbf{z}_k \sim \mathcal{N}(\mathbf{0},\mathbf{I}).
$$

**Exercise 3.** Write `langevin(score_fn, x0, eps, n_steps)`, which runs `n_steps` steps from the batch of starting points `x0` and returns the final points.

In [ ]:
def langevin(score_fn, x0, eps, n_steps):
    '''score_fn: (n, 2) -> (n, 2); x0: (n, 2). Returns the final points, shape (n, 2).'''
    # TODO
    raise NotImplementedError

In [ ]:
check_langevin(langevin)   # with the score of a standard Gaussian, -x, Langevin should give standard Gaussian samples

We start from points spread uniformly over the square $[-8, 8]^2$, as in Song & Ermon (2019).

In [ ]:
N_SAMPLES = 2000
x0 = 16 * torch.rand(N_SAMPLES, 2) - 8

x_langevin = langevin(true_score, x0, eps=0.1, n_steps=1000)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
plot_samples(x_data[:N_SAMPLES], ax=axes[0], title=f"data: {fraction_small_mode(x_data):.0%} in the small mode")
plot_samples(x_langevin, ax=axes[1], title=f"Langevin: {fraction_small_mode(x_langevin):.0%} in the small mode")
plt.show()

**Questions.**

1. What proportion of the samples ends in the small mode? Why is it close to the proportion of *starting points* on that side, rather than to $\pi_1 = 0.2$?
2. Run 10 times more steps (`n_steps=10_000`). Does it help? Why?
3. What would happen with the starting points all in the top-right corner? How could you check it with an experiment?

## 4. Annealed Langevin dynamics with the exact noisy scores

We use $L = 10$ noise levels in a geometric progression, $\sigma_1 = 1 < \dots < \sigma_{10} = 20$, and run Langevin dynamics at each level, from the largest to the smallest, with a step size that grows with the noise (the lecture's rule, with $\epsilon$ the step size at $\sigma = 1$; it is the same as in the lecture when $\sigma_1 = 1$):
$$
\alpha_i = \epsilon\,\sigma_i^2,
\qquad
\mathbf{x} \leftarrow \mathbf{x} + \frac{\alpha_i}{2}\,\nabla_\mathbf{x}\log p_{\sigma_i}(\mathbf{x}) + \sqrt{\alpha_i}\,\mathbf{z}.
$$

**Exercise 4.** Complete `annealed_langevin(score_fn, x0, sigmas, eps, n_steps)`, where `score_fn(x, sigma)` returns the score of $p_\sigma$, `sigmas` is sorted in increasing order, and `n_steps` is the number of steps *per noise level*. The loop over the noise levels, from the largest to the smallest, the step size $\alpha_i$ and the record of the samples after each level are given: write one Langevin step (the line marked `TODO`).

In [ ]:
SIGMAS = torch.logspace(math.log10(1.0), math.log10(20.0), 10)   # sigma_1 < ... < sigma_L
print(SIGMAS)


def annealed_langevin(score_fn, x0, sigmas, eps, n_steps):
    '''score_fn: (x of shape (n, 2), sigma a number) -> (n, 2); x0: (n, 2); sigmas: increasing.
    Returns the final points (n, 2), and a list of (sigma, points) after each level.'''
    x = x0.clone()                      # work on a copy: x0 is reused by the other samplers
    history = []
    for sigma in reversed(sigmas.tolist()):                      # large to small
        alpha = eps * sigma**2                                   # step size: grows with the noise
        # The lecture writes alpha = eps * sigma**2 / sigma_1**2, with eps the step at the smallest level sigma_1.
        # Here eps is the step at sigma = 1 (the same thing when sigma_1 = 1): it does not depend on the choice
        # of noise levels, so it needs no retuning when they change.
        for _ in range(n_steps):
            x = ...                                              # TODO: one Langevin step at noise level sigma, with step size alpha
        history.append((sigma, x.clone()))                       # samples after this level
    return x, history

In [ ]:
check_annealed_langevin(annealed_langevin, SIGMAS)

In [ ]:
x_annealed, history = annealed_langevin(true_score, x0, SIGMAS, eps=0.1, n_steps=100)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, (sigma, xs) in zip(axes, [history[0], history[3], history[6], history[-1]]):
    plot_samples(xs, ax=ax, sigma=sigma,
                 title=f"after $\\sigma$ = {sigma:.1f}: {fraction_small_mode(xs):.0%} on the small-mode side")
plt.show()

In [ ]:
# How close are the samples to p? Wasserstein distance (on the side of the large mode) to fresh samples from p
x_ref = sample_data(N_SAMPLES)
print("Wasserstein, annealed Langevin (sigma_1 = 1) vs data:", wasserstein_large(x_annealed, x_ref))

**Questions.**

1. What proportion of the samples ends in the small mode now?
2. The samples are drawn from $p_{\sigma_1}$, not from $p$. With $\sigma_1 = 1$, how different are they? How would you get closer to $p$? How would you implement it? To test your solution, measure how close your samples are to $p$ with `wasserstein_large(x, y)` (in `utils_score_matching.py`), as in the cell above: the Wasserstein distance between two sets of samples, on the side of the large mode, here between your samples and fresh samples from `sample_data`.

## 5. Learning the score with denoising score matching

So far we have cheated: we used the exact score. Now we only use the training set `x_data`, and learn the score with a small neural network.

### 5.1 A single noise level

With one noise level $\sigma$, the denoising loss from the lecture is
$$
\mathbb{E}_{\mathbf{x}\sim p,\ \mathbf{z}\sim\mathcal{N}(\mathbf{0},\mathbf{I})}
\Big\|\, s_\theta(\mathbf{x}+\sigma\mathbf{z}) + \frac{\mathbf{z}}{\sigma} \Big\|^2,
$$
and its minimiser is the score of the noisy data, $\nabla_\mathbf{x} \log p_\sigma$.

**Exercise 5.** We need the same small network twice: with 2 inputs here, and with 3 inputs ($\mathbf{x}$ and $\log\sigma$) in Section 5.2. Write `make_mlp(d_in, width=128)`, which returns it as an `nn.Sequential`: 3 hidden layers of `width` units, each followed by a SiLU activation, then a last linear layer with 2 outputs (the score) and no activation (the score can be any vector).

*Why SiLU?* $\mathrm{SiLU}(x) = x \cdot \mathrm{sigmoid}(x)$ is a smooth version of ReLU. It is the activation used in the U-Nets of diffusion models such as DDPM and Stable Diffusion. A smooth activation gives a smooth learned score, whereas a ReLU network is piecewise linear, with kinks.

*Hint:* `nn.Sequential(nn.Linear(d_in, width), nn.SiLU(), ...)`.

In [ ]:
def make_mlp(d_in, width=128):
    '''Returns an nn.Sequential: (B, d_in) -> (B, 2).'''
    # TODO
    raise NotImplementedError

In [ ]:
check_make_mlp(make_mlp)

**Exercise 6.** Write `dsm_loss(model, x, sigma)`: for a minibatch `x` of clean data, draw the noise, build the noisy points, and return the denoising loss, averaged over the minibatch (a single number).

*Shapes:* `x`, `z` and `model(x_noisy)` are all `(B, 2)`; sum the squares over the 2 coordinates (`dim=1`), then average over the batch.

In [ ]:
def dsm_loss(model, x, sigma):
    '''model: (B, 2) -> (B, 2); x: (B, 2); sigma: a number. Returns a scalar.'''
    # TODO
    raise NotImplementedError

In [ ]:
check_dsm_loss(dsm_loss)   # the exact score must do much better than predicting 0

In [ ]:
def train_single_level(sigma, n_steps=3000, batch_size=512, lr=1e-3):
    model = make_mlp(2)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for step in range(n_steps):
        x = x_data[torch.randint(len(x_data), (batch_size,))]       # (B, 2)
        loss = dsm_loss(model, x, sigma)
        opt.zero_grad(); loss.backward(); opt.step()
        if step % 1000 == 0:
            print(f"step {step:5d}  loss {loss.item():.3f}")
    return model


SIGMA_SINGLE = 1.0
model_single = train_single_level(SIGMA_SINGLE)

Compare the learned field with the true score of $p_\sigma$, and look at where the error is large (`plot_learned_score`, from `utils_score_matching.py`, draws the true score, the learned score and the size of the error):

In [ ]:
plot_learned_score(model_single, true_score, SIGMA_SINGLE, x_data)

**Questions.**

1. Where is the learned score accurate, and where is it wrong? Relate this to the slide "Problem 1: no data, no arrows".
2. Sample with plain Langevin dynamics using `model_single` instead of `true_score` (run the cell below). What goes wrong?

In [ ]:
with torch.no_grad():
    x_single = langevin(model_single, x0, eps=0.1, n_steps=1000)
plot_samples(x_single, title=f"Langevin, learned score: {fraction_small_mode(x_single):.0%} in the small mode")
plt.show()

### 5.2 A noise-conditional score network (NCSN)

We now learn one network $s_\theta(\mathbf{x}, \sigma)$ for all the noise levels `SIGMAS`, by minimising the objective of the lecture:
$$
\mathcal{J}(\theta) = \mathbb{E}_{i\sim\mathcal{U}\{1,\dots,L\}}\;\mathbb{E}_{\mathbf{x},\mathbf{z}}\,
\big\|\,\sigma_i\, s_\theta(\mathbf{x}+\sigma_i\mathbf{z},\,\sigma_i) + \mathbf{z}\,\big\|^2,
$$
i.e. the weighted sum of the denoising losses with weights $w_i = \sigma_i^2$.

The network gets $\log\sigma$ as an extra input. We also divide its output by $\sigma$, since the true score has size about $1/\sigma$ (this is the trick of the improved NCSN, Song & Ermon 2020): the network then only has to output something of size about 1.

In [ ]:
class NCSN(nn.Module):
    def __init__(self, width=128):
        super().__init__()
        self.mlp = make_mlp(3, width=width)

    def forward(self, x, sigma):
        '''x: (B, 2); sigma: a number, or a tensor of shape (B, 1). Returns (B, 2).'''
        sigma = torch.as_tensor(sigma, dtype=x.dtype)
        if sigma.ndim == 0:
            sigma = sigma.expand(x.shape[0], 1)                     # same sigma for every point
        return self.mlp(torch.cat([x, sigma.log()], dim=1)) / sigma

We train as in the slides, but with the gradient averaged over a minibatch, and a noise level drawn for each example: the update is the same in expectation, with much less noise.

**Exercise 7.** Write `ncsn_loss(model, x, sigmas)`: draw a noise level *for each example* uniformly among `sigmas`, then return the loss $\mathcal{J}$ on the minibatch.

*Shapes:* keep `sigma` of shape `(B, 1)` everywhere. Then `sigma * z` is `(B, 2)` (each row is multiplied by its own noise level), and `model(x_noisy, sigma)` accepts it directly.
*Hint:* `torch.randint(len(sigmas), (B, 1))` gives random indices of shape `(B, 1)`, and `sigmas[indices]` has the same shape.

In [ ]:
def ncsn_loss(model, x, sigmas):
    '''model: (x of shape (B, 2), sigma of shape (B, 1)) -> (B, 2); x: (B, 2); sigmas: (L,).
    Returns a scalar.'''
    # TODO
    raise NotImplementedError

In [ ]:
check_ncsn_loss(ncsn_loss, SIGMAS)

In [ ]:
def train_ncsn(sigmas, n_steps=5000, batch_size=512, lr=1e-3):
    model = NCSN()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for step in range(n_steps):
        x = x_data[torch.randint(len(x_data), (batch_size,))]       # (B, 2)
        loss = ncsn_loss(model, x, sigmas)
        opt.zero_grad(); loss.backward(); opt.step()
        if step % 1000 == 0:
            print(f"step {step:5d}  loss {loss.item():.3f}")
    return model


model_ncsn = train_ncsn(SIGMAS)

Now sample with annealed Langevin dynamics, using the learned network instead of the exact scores:

In [ ]:
with torch.no_grad():
    x_ncsn, history_ncsn = annealed_langevin(model_ncsn, x0, SIGMAS, eps=0.1, n_steps=100)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
plot_samples(x_data[:N_SAMPLES], ax=axes[0], title=f"data: {fraction_small_mode(x_data):.0%} in the small mode")
plot_samples(x_langevin, ax=axes[1], title=f"Langevin, exact score: {fraction_small_mode(x_langevin):.0%}")
plot_samples(x_ncsn, ax=axes[2], title=f"annealed Langevin, NCSN: {fraction_small_mode(x_ncsn):.0%}")
plt.show()

The cell below compares the learned score with the true score at the four smallest noise levels and at the largest one (`plot_ncsn_scores`, in `utils_score_matching.py`). The error uses the same colour scale on the three rows, so that the levels can be compared:

In [ ]:
plot_ncsn_scores(model_ncsn, true_score, [SIGMAS[i].item() for i in [0, 1, 2, 3, -1]], x_data)   # the 4 smallest levels and the largest

**Questions.**

1. Does the learned NCSN recover the right proportions?
2. Look at the comparisons of the learned and true scores above, from the smallest noise level to the largest. Where is the learned score accurate, and how does this change with the noise level?

## 6. Going further (optional)

1. **Weights.** Retrain the NCSN with uniform weights $w_i = 1$, i.e. with the loss $\|s_\theta(\tilde{\mathbf{x}}, \sigma_i) + \mathbf{z}/\sigma_i\|^2$. Compare the error of the learned score at each noise level with the one obtained with $w_i = \sigma_i^2$. Which levels are learned worse, and why?
2. **Too little noise.** Replace `SIGMAS` by levels between 0.1 and 2. What happens to the proportions? Why?
3. **Unbalanced data.** Change the weights to $\pi = (0.05, 0.95)$ with `set_weights([0.05, 0.95])` (this updates `PIS` everywhere, including in the helpers), then rerun the notebook from Section 1, without the first cell. Does annealed Langevin still get the proportions right?
4. **The score with autograd.** The score is the gradient of the log-density, so it can also be computed by automatic differentiation. Write `autograd_score(x, sigma)` using `log_density(x, sigma)` (in `utils_score_matching.py`) and `torch.autograd.grad`, and check that it matches your `true_score`. *(Hint: `x = x.clone().requires_grad_(True)`; differentiate `log_density(x, sigma).sum()`: why the sum?)*
